In [ ]:
import pybamm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math
import dfols
import signal
from tqdm import tqdm
from scipy.integrate import solve_ivp
from scipy.fft import fft, fftfreq, fftshift
from scipy.signal import savgol_filter
from scipy.signal import find_peaks
from scipy import interpolate, integrate
from stopit import threading_timeoutable as timeoutable
import os, sys
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))
from batfuns import *
plt.rcParams = set_rc_params(plt.rcParams)
import winsound
from pybamm import exp, constants, Parameter
import pickle

pd.options.mode.chained_assignment = None

eSOH_DIR = "../data/esoh_R/"
oCV_DIR = "../data/ocv/"
cyc_DIR = "../data/cycling/"
fig_DIR = "../figures/figures_fit/"
res_DIR = "../data/results_paper/"
resistance_DIR = "../data/resistance/"
%matplotlib widget

In [ ]:
def get_cdf(cell,n=0):
    cell_no = f'{cell:02d}'
    data = pd.read_csv(cyc_DIR+"cycling_data_cell_"+cell_no+".csv")
    N1 = np.unique(data["Cycle number"])
    print(N1[-1])
    df = data[data["Cycle number"]==N1[n]]
    df["Time [s]"] = df["Time [s]"] - df["Time [s]"].iloc[0]
    df["Q [Ah]"] = df["Q [Ah]"] - df["Q [Ah]"].iloc[0]
    return df

In [ ]:
cell = 26
cell_no = f'{cell:02d}'
data = pd.read_csv(cyc_DIR+"cycling_data_cell_"+cell_no+".csv")
N1 = np.unique(data["Cycle number"])
n = 1
df0 = data[data["Cycle number"]==N1[n]]
df0["Time [s]"] = df0["Time [s]"] - df0["Time [s]"].iloc[0]
cell = 28
cell_no = f'{cell:02d}'
data = pd.read_csv(cyc_DIR+"cycling_data_cell_"+cell_no+".csv")
N1 = np.unique(data["Cycle number"])
n = 0
df5 = data[data["Cycle number"]==N1[n]]

In [ ]:
df0  = get_cdf(26,n=1)
df1  = get_cdf(28,n=1)
df2  = get_cdf(29,n=1)
df3  = get_cdf(31,n=1)
df4  = get_cdf(33,n=1)
df5  = get_cdf(35,n=1)

In [ ]:
colors = ['blue','red','green','orange','magenta','cyan']
fig, ax = plt.subplots(1,1,figsize=(5,4))
ax.plot(df0["Time [s]"],df0["Voltage [V]"],color=colors[0],linewidth=1.5)
ax.plot(df1["Time [s]"],df1["Voltage [V]"],color=colors[1],linewidth=1.5)
ax.plot(df2["Time [s]"],df2["Voltage [V]"],color=colors[2],linewidth=1.5)
ax.plot(df3["Time [s]"],df3["Voltage [V]"],color=colors[3],linewidth=1.5)
ax.plot(df4["Time [s]"],df4["Voltage [V]"],color=colors[4],linewidth=1.5)
ax.plot(df5["Time [s]"],df5["Voltage [V]"],color=colors[5],linewidth=1.5)
ax.legend(['0 psi','5 psi','10 psi','15 psi','20 psi','25 psi','sim'])
ax.set_xlabel("Time [s]")
ax.set_ylabel("Cycling Voltage [V]")
ax.set_title("Fresh Cell Voltage for Various Pressures")
fig.savefig(fig_DIR + "BOL_V_Pressure.png")

In [ ]:
df0  = get_cdf(26,n=75)
df1  = get_cdf(28,n=75)
df2  = get_cdf(29,n=75)
df3  = get_cdf(31,n=75)
df4  = get_cdf(33,n=75)
df5  = get_cdf(35,n=75)

In [ ]:
colors = ['blue','red','green','orange','magenta','cyan']
fig, ax = plt.subplots(1,1,figsize=(5,4))
ax.plot(df0["Time [s]"],df0["Voltage [V]"],color=colors[0],linewidth=1.5)
ax.plot(df1["Time [s]"],df1["Voltage [V]"],color=colors[1],linewidth=1.5)
ax.plot(df2["Time [s]"],df2["Voltage [V]"],color=colors[2],linewidth=1.5)
ax.plot(df3["Time [s]"],df3["Voltage [V]"],color=colors[3],linewidth=1.5)
ax.plot(df4["Time [s]"],df4["Voltage [V]"],color=colors[4],linewidth=1.5)
ax.plot(df5["Time [s]"],df5["Voltage [V]"],color=colors[5],linewidth=1.5)
ax.legend(['0 psi','5 psi','10 psi','15 psi','20 psi','25 psi','sim'])
ax.set_xlabel("Time [s]")
ax.set_ylabel("Cycling Voltage [V]")
ax.set_title("MOL (Cyc=75) Voltage for Various Pressures")
fig.savefig(fig_DIR + "MOL_V_Pressure.png")

In [ ]:
df0  = get_cdf(26,n=150)
df1  = get_cdf(28,n=150)
df2  = get_cdf(29,n=150)
df3  = get_cdf(31,n=150)
df4  = get_cdf(33,n=150)
df5  = get_cdf(35,n=150)

In [ ]:
colors = ['blue','red','green','orange','magenta','cyan']
fig, ax = plt.subplots(1,1,figsize=(5,4))
ax.plot(df0["Time [s]"],df0["Voltage [V]"],color=colors[0],linewidth=1.5)
ax.plot(df1["Time [s]"],df1["Voltage [V]"],color=colors[1],linewidth=1.5)
ax.plot(df2["Time [s]"],df2["Voltage [V]"],color=colors[2],linewidth=1.5)
ax.plot(df3["Time [s]"],df3["Voltage [V]"],color=colors[3],linewidth=1.5)
ax.plot(df4["Time [s]"],df4["Voltage [V]"],color=colors[4],linewidth=1.5)
ax.plot(df5["Time [s]"],df5["Voltage [V]"],color=colors[5],linewidth=1.5)
ax.legend(['0 psi','5 psi','10 psi','15 psi','20 psi','25 psi','sim'])
ax.set_xlabel("Time [s]")
ax.set_ylabel("Cycling Voltage [V]")
ax.set_title("MOL (Cyc=150) Voltage for Various Pressures")
fig.savefig(fig_DIR + "EOL_V_Pressure.png")

In [ ]:
n = 25
cell = 33
cell_no = f'{cell:02d}'
data = pd.read_csv(cyc_DIR+"cycling_data_cell_"+cell_no+".csv")
N1 = np.unique(data["Cycle number"])
print(N1[-1])
df = data[data["Cycle number"]==N1[n]]
df["Time [s]"] = df["Time [s]"] - df["Time [s]"].iloc[0]
df["Q [Ah]"] = df["Q [Ah]"] - df["Q [Ah]"].iloc[0]
t = df["Time [s]"].to_numpy()
I = df["Current [mA]"].to_numpy()/1000
V = df["Voltage [V]"].to_numpy()
Q = df["Q [Ah]"].to_numpy()
Cap = df["Capacity [Ah]"].to_numpy()
idxi1 = np.where((np.diff(I)<-1) & (I[:-1]>0))[0]
idx = idxi1[0]
t_ch = t[:idx]
I_ch = I[:idx]
V_ch = V[:idx]
Q_ch = Cap[:idx]
t_dh = t[idx:]-t[idx]
I_dh = I[idx:]
V_dh = V[idx:]
Q_dh = Cap[idx:]
fig,ax = plt.subplots(1,1)
ax.plot(t,I)
ax.plot(t[idx],I[idx],'x')

In [ ]:
def get_chdh(cell,n=0):
    cell_no = f'{cell:02d}'
    data = pd.read_csv(cyc_DIR+"cycling_data_cell_"+cell_no+".csv")
    N1 = np.unique(data["Cycle number"])
    # print(N1[-1])
    df = data[data["Cycle number"]==N1[n]]
    if n == 1 and cell == 31:
        df = df.drop(df.index[0])
    df["Time [s]"] = df["Time [s]"] - df["Time [s]"].iloc[0]
    df["Q [Ah]"] = df["Q [Ah]"] - df["Q [Ah]"].iloc[0]
    t = df["Time [s]"].to_numpy()
    I = df["Current [mA]"].to_numpy()/1000
    V = df["Voltage [V]"].to_numpy()
    Q = df["Q [Ah]"].to_numpy()
    Cap = df["Capacity [Ah]"].to_numpy()
    idxi1 = np.where((np.diff(I)<-1) & (I[:-1]>0) & (I[:-1]<2))[0]
    idx = idxi1[0]
    t_ch = t[:idx-1]
    I_ch = I[:idx-1]
    V_ch = V[:idx-1]
    Q_ch = Cap[:idx-1]
    t_dh = t[idx+1:]-t[idx+1]
    I_dh = I[idx+1:]
    V_dh = V[idx+1:]
    Q_dh = Cap[idx+1:]
    return t_ch,I_ch,V_ch,Q_ch,t_dh,I_dh,V_dh,Q_dh

In [ ]:
cell = 31
n = 1
cell_no = f'{cell:02d}'
data = pd.read_csv(cyc_DIR+"cycling_data_cell_"+cell_no+".csv")
N1 = np.unique(data["Cycle number"])
# print(N1[-1])
df = data[data["Cycle number"]==N1[n]]
if n == 1 and cell == 31:
    df = df.drop(df.index[0])
df["Time [s]"] = df["Time [s]"] - df["Time [s]"].iloc[0]
df["Q [Ah]"] = df["Q [Ah]"] - df["Q [Ah]"].iloc[0]
t = df["Time [s]"].to_numpy()
I = df["Current [mA]"].to_numpy()/1000
V = df["Voltage [V]"].to_numpy()
Q = df["Q [Ah]"].to_numpy()
Cap = df["Capacity [Ah]"].to_numpy()
idxi1 = np.where((np.diff(I)<-1) & (I[:-1]>0) & (I[:-1]<2))[0]
idx = idxi1[0]
t_ch = t[:idx-1]
I_ch = I[:idx-1]
V_ch = V[:idx-1]
Q_ch = Cap[:idx-1]
t_dh = t[idx+1:]-t[idx+1]
I_dh = I[idx+1:]
V_dh = V[idx+1:]
Q_dh = Cap[idx+1:]

In [ ]:
for n in [1,25,50,75,100,125,150]:
# for n in [1]:
    t_ch0,I_ch0,V_ch0,Q_ch0,t_dh0,I_dh0,V_dh0,Q_dh0  = get_chdh(26,n=n)
    t_ch1,I_ch1,V_ch1,Q_ch1,t_dh1,I_dh1,V_dh1,Q_dh1  = get_chdh(28,n=n)
    t_ch2,I_ch2,V_ch2,Q_ch2,t_dh2,I_dh2,V_dh2,Q_dh2  = get_chdh(29,n=n)
    t_ch3,I_ch3,V_ch3,Q_ch3,t_dh3,I_dh3,V_dh3,Q_dh3  = get_chdh(31,n=n)
    t_ch4,I_ch4,V_ch4,Q_ch4,t_dh4,I_dh4,V_dh4,Q_dh4  = get_chdh(33,n=n)
    t_ch5,I_ch5,V_ch5,Q_ch5,t_dh5,I_dh5,V_dh5,Q_dh5  = get_chdh(35,n=n)
    # cy = 1
    colors = ['blue','red','green','orange','magenta','cyan']
    fig, ax = plt.subplots(1,1,figsize=(5,4))
    ax.plot(Q_ch0,V_ch0,color=colors[0],linewidth=1.5)
    ax.plot(Q_ch1,V_ch1,color=colors[1],linewidth=1.5)
    ax.plot(Q_ch2,V_ch2,color=colors[2],linewidth=1.5)
    ax.plot(Q_ch3,V_ch3,color=colors[3],linewidth=1.5)
    ax.plot(Q_ch4,V_ch4,color=colors[4],linewidth=1.5)
    ax.plot(Q_ch5,V_ch5,color=colors[5],linewidth=1.5)
    ax.legend(['0 psi','5 psi','10 psi','15 psi','20 psi','25 psi','sim'])
    ax.set_xlim([0,5])
    ax.set_xlabel("Q [Ah]")
    ax.set_ylabel("Cycling Voltage [V]")
    ax.set_title(f"Voltage for Various Pressures: N={n}")
    fig.savefig(fig_DIR + f"Pressure_data_{n}_ch.png")

In [ ]:
for n in [1,25,50,75,100,125,150]:
# for n in [1]:
    t_ch0,I_ch0,V_ch0,Q_ch0,t_dh0,I_dh0,V_dh0,Q_dh0  = get_chdh(26,n=n)
    t_ch1,I_ch1,V_ch1,Q_ch1,t_dh1,I_dh1,V_dh1,Q_dh1  = get_chdh(28,n=n)
    t_ch2,I_ch2,V_ch2,Q_ch2,t_dh2,I_dh2,V_dh2,Q_dh2  = get_chdh(29,n=n)
    t_ch3,I_ch3,V_ch3,Q_ch3,t_dh3,I_dh3,V_dh3,Q_dh3  = get_chdh(31,n=n)
    t_ch4,I_ch4,V_ch4,Q_ch4,t_dh4,I_dh4,V_dh4,Q_dh4  = get_chdh(33,n=n)
    t_ch5,I_ch5,V_ch5,Q_ch5,t_dh5,I_dh5,V_dh5,Q_dh5  = get_chdh(35,n=n)
    # cy = 1
    colors = ['blue','red','green','orange','magenta','cyan']
    fig, ax = plt.subplots(1,1,figsize=(5,4))
    ax.plot(Q_dh0,V_dh0,color=colors[0],linewidth=1.5)
    ax.plot(Q_dh1,V_dh1,color=colors[1],linewidth=1.5)
    ax.plot(Q_dh2,V_dh2,color=colors[2],linewidth=1.5)
    ax.plot(Q_dh3,V_dh3,color=colors[3],linewidth=1.5)
    ax.plot(Q_dh4,V_dh4,color=colors[4],linewidth=1.5)
    ax.plot(Q_dh5,V_dh5,color=colors[5],linewidth=1.5)
    ax.legend(['0 psi','5 psi','10 psi','15 psi','20 psi','25 psi','sim'])
    ax.set_xlim([0,5])
    ax.set_xlabel("Q [Ah]")
    ax.set_ylabel("Cycling Voltage [V]")
    ax.set_title(f"Voltage for Various Pressures: N={n}")
    fig.savefig(fig_DIR + f"Pressure_data_{n}_dh.png")

## Expansion

In [ ]:
def get_exp_peaks(cell):
    cell_no = f'{cell:02d}'
    data = pd.read_csv(cyc_DIR+"cycling_data_cell_"+cell_no+".csv")
    N1 = np.unique(data["Cycle number"])
    # print(N1[-1])
    dfa = []
    for n in N1:
        try:
            df = data[data["Cycle number"]==N1[n]]
            df = df.dropna()
            df["Time [s]"] = df["Time [s]"] - df["Time [s]"].iloc[0]
            df["Q [Ah]"] = df["Q [Ah]"] - df["Q [Ah]"].iloc[0]
            t = df["Time [s]"].to_numpy()
            I = df["Current [mA]"].to_numpy()/1000
            V = df["Voltage [V]"].to_numpy()
            Q = df["Q [Ah]"].to_numpy()
            Cap = df["Capacity [Ah]"].to_numpy()
            E = df["Expansion [mu m]"].to_numpy()
            E = E-E[0]
        
            idxi1 = np.where((np.diff(I)<-1) & (I[:-1]>0) & (I[:-1]<2))[0]
            idx = idxi1[0]
            t_ch = t[:idx-1]
            I_ch = I[:idx-1]
            V_ch = V[:idx-1]
            Q_ch = Cap[:idx-1]
            E_ch = E[:idx-1]
            t_dh = t[idx+1:]-t[idx+1]
            I_dh = I[idx+1:]
            V_dh = V[idx+1:]
            Q_dh = Cap[idx+1:]
            E_dh = E[idx+1:]
            window_length=61
            polyorder=3
            if len(t)>1000:
                window_length=501
            S_ch = Q_ch/max(Q_ch)
            dQ = savgol_filter(Q_ch,window_length,polyorder,1)
            dE2 = savgol_filter(E_ch,window_length,polyorder,2)
            dE2 = dE2*(S_ch<=0.95)
            dEdQ2 = dE2/dQ/dQ
            pks_,_ = find_peaks(-dEdQ2,height=0.01,width=0.1)
            pks1 = [pk for pk in pks_ if S_ch[pk]>=0.1 and S_ch[pk]<=0.9]
            pks_,_ = find_peaks(dEdQ2,height=0.01,width=0.1)
            pks2 = [pk for pk in pks_ if S_ch[pk]>=0.1 and S_ch[pk]<=0.9]
            pks = np.append(pks1,pks2)
        
            p1s = np.round(S_ch[pks1[0]],4)
            p2s = np.round(S_ch[pks2[0]],4)
            p1q = np.round(Q_ch[pks1[0]],3)
            p2q = np.round(Q_ch[pks2[0]],3)
            Qmax = np.round(Q_ch[-1],3)
            df1 = pd.DataFrame({"N":n+1,"p1s":p1s,"p2s":p2s,"p1q":p1q,"p2q":p2q,"Qch":Qmax},index=[0])
            df1["ds"] = df1["p2s"] - df1["p1s"]
            df1["dq"] = df1["p2q"] - df1["p1q"]
            dfa.append(df1)
        except:
            pass
    df2 = pd.concat(dfa)
    return df2

In [ ]:
df0 = get_exp_peaks(4)
df1 = get_exp_peaks(6)
df2 = get_exp_peaks(7)
df3 = get_exp_peaks(9)

In [ ]:
df4 = get_exp_peaks(1)
df5 = get_exp_peaks(3)

In [ ]:
_,_,dfe0,_,_,_ = load_data(4,eSOH_DIR,oCV_DIR)
_,_,dfe1,_,_,_ = load_data(6,eSOH_DIR,oCV_DIR)
_,_,dfe2,_,_,_ = load_data(7,eSOH_DIR,oCV_DIR)
_,_,dfe3,_,_,_ = load_data(9,eSOH_DIR,oCV_DIR)
_,_,dfe4,_,_,_ = load_data(1,eSOH_DIR,oCV_DIR)
_,_,dfe5,_,_,_ = load_data(3,eSOH_DIR,oCV_DIR)

In [ ]:
colors = ['blue','red','green','orange','magenta','cyan']
fig, ax = plt.subplots(1,1,figsize=(6.2,4.8))
ax.plot(df0["N"],df0["dq"]/(0.5-0.24),color=colors[0],linewidth=1.5)
ax.plot(dfe0["N"],dfe0["C_n"],'o',color=colors[3],)
ax.legend(['Cn from E','Cn from V'])
# ax.legend(['0 psi','10 psi','15 psi','20 psi','25 psi','sim'])
# ax.set_ylim([0.5,2])
ax.set_xlabel("Cycle Number",fontsize=16)
ax.set_ylabel(r"$C_n$",fontsize=16)
ax.set_title(r"Room 1.5C",fontsize=16)
fig.savefig(fig_DIR + f"Cn_Comparison_1p5C_room.png")

In [ ]:
colors = ['blue','red','green','orange','magenta','cyan']
fig, ax = plt.subplots(1,1,figsize=(6.2,4.8))
ax.plot(df1["N"],df1["dq"]/(0.5-0.24),color=colors[1],linewidth=1.5)
ax.plot(dfe1["N"],dfe1["C_n"],'o',color=colors[2],)
ax.legend(['Cn from E','Cn from V'])
# ax.legend(['0 psi','10 psi','15 psi','20 psi','25 psi','sim'])
# ax.set_ylim([0.5,2])
ax.set_xlabel("Cycle Number",fontsize=16)
ax.set_ylabel(r"$C_n$",fontsize=16)
ax.set_title(r"Hot 1.5C",fontsize=16)
fig.savefig(fig_DIR + f"Cn_Comparison_1p5C_hot.png")

In [ ]:
colors = ['blue','red','green','orange','magenta','cyan']
fig, ax = plt.subplots(1,1,figsize=(6.2,4.8))
ax.plot(df2["N"],df2["dq"]/(0.5-0.24),color=colors[2],linewidth=1.5)
ax.plot(dfe2["N"],dfe2["C_n"],'o',color=colors[1],)
ax.legend(['Cn from E','Cn from V'])
# ax.legend(['0 psi','10 psi','15 psi','20 psi','25 psi','sim'])
# ax.set_ylim([0.5,2])
ax.set_xlabel("Cycle Number",fontsize=16)
ax.set_ylabel(r"$C_n$",fontsize=16)
ax.set_title(r"Room 2C",fontsize=16)
fig.savefig(fig_DIR + f"Cn_Comparison_2C_room.png")


In [ ]:
colors = ['blue','red','green','orange','magenta','cyan']
fig, ax = plt.subplots(1,1,figsize=(6.2,4.8))
ax.plot(df3["N"],df3["dq"]/(0.5-0.24),color=colors[3],linewidth=1.5)
ax.plot(dfe3["N"],dfe3["C_n"],'o',color=colors[0],)
ax.legend(['Cn from E','Cn from V'])
# ax.legend(['0 psi','10 psi','15 psi','20 psi','25 psi','sim'])
# ax.set_ylim([0.5,2])
ax.set_xlabel("Cycle Number",fontsize=16)
ax.set_ylabel(r"$C_n$",fontsize=16)
ax.set_title(r"Hot 2C",fontsize=16)
fig.savefig(fig_DIR + f"Cn_Comparison_2C_hot.png")


In [ ]:
sdfsd

In [ ]:

colors = ['blue','red','green','orange','magenta','cyan']
fig, ax = plt.subplots(1,1,figsize=(6.2,4.8))
ax.plot(df0["N"],df0["dq"]/(0.5-0.24),color=colors[0],linewidth=1.5)
ax.plot(dfe0["N"],dfe0["C_n"],'o',color=colors[0],)
ax.plot(df1["N"],df1["dq"]/(0.5-0.24),color=colors[1],linewidth=1.5)
ax.plot(dfe1["N"],dfe1["C_n"],'o',color=colors[1],)
ax.plot(df2["N"],df2["dq"]/(0.5-0.24),color=colors[2],linewidth=1.5)
ax.plot(dfe2["N"],dfe2["C_n"],'o',color=colors[2],)
ax.plot(df3["N"],df3["dq"]/(0.5-0.24),color=colors[3],linewidth=1.5)
ax.plot(dfe3["N"],dfe3["C_n"],'o',color=colors[3],)

ax.legend(['Room 1.5C: E','Room 1.5C: V','Hot 1.5C: E','Hot 1.5C: V','Room 2C: E','Room 2C: V','Hot 2C: E','Hot 2C: V'])
# ax.legend(['0 psi','10 psi','15 psi','20 psi','25 psi','sim'])
# ax.set_ylim([0.5,2])
ax.set_xlabel("Cycle Number",fontsize=16)
ax.set_ylabel(r"$C_n$",fontsize=16)
ax.set_title(r"$C_n$: $\frac{Q_{p2}-Q_{p1}}{x_{p2}-x_{p1}}$",fontsize=16)
fig.savefig(fig_DIR + f"Expansion_Cn_Comparison.png")

In [ ]:
df0 = get_exp_peaks(26)
df1 = get_exp_peaks(9)
df2 = get_exp_peaks(29)
df3 = get_exp_peaks(31)
df4 = get_exp_peaks(33)
df5 = get_exp_peaks(35)
colors = ['blue','red','green','orange','magenta','cyan']
fig, ax = plt.subplots(1,1,figsize=(6.2,4.8))
ax.plot(df0["N"],df0["dq"]/(0.5-0.24),color=colors[0],linewidth=1.5)
ax.plot(df1["N"],df1["dq"]/(0.5-0.24),color=colors[1],linewidth=1.5)
ax.plot(df2["N"],df2["dq"]/(0.5-0.24),color=colors[2],linewidth=1.5)
ax.plot(df3["N"],df3["dq"]/(0.5-0.24),color=colors[3],linewidth=1.5)
ax.plot(df4["N"],df4["dq"]/(0.5-0.24),color=colors[4],linewidth=1.5)
ax.plot(df5["N"],df5["dq"]/(0.5-0.24),color=colors[5],linewidth=1.5)
ax.legend(['0 psi','5 psi','10 psi','15 psi','20 psi','25 psi','sim'])
# ax.legend(['0 psi','10 psi','15 psi','20 psi','25 psi','sim'])
# ax.set_ylim([0.5,2])
ax.set_xlabel("Cycle Number",fontsize=16)
ax.set_ylabel(r"$C_n$",fontsize=16)
ax.set_title(r"$C_n$: $\frac{Q_{p2}-Q_{p1}}{x_{p2}-x_{p1}}$",fontsize=16)
fig.savefig(fig_DIR + f"Expansion_Cn_Pressure.png")

In [ ]:
asdas

In [ ]:
df6 = get_exp_peaks(29)
colors = ['blue','red','green','orange','magenta','cyan']
fig, ax = plt.subplots(1,1,figsize=(10,8))
ax.plot(df6["N"],df6["dq"]/(0.5-0.24),color=colors[0],linewidth=1.5)
# ax.set_ylim([0.5,2])
ax.set_xlabel("Cycle Number")
ax.set_ylabel(r"$\Delta SOC$")
ax.set_title(r"Peaks: $\Delta SOC$")
# fig.savefig(fig_DIR + f"Pressure_data_{n}_dh.png")

In [ ]:
cell = 7
n = 120
cell_no = f'{cell:02d}'
data = pd.read_csv(cyc_DIR+"cycling_data_cell_"+cell_no+".csv")
N1 = np.unique(data["Cycle number"])
# print(N1[-1])
fig,ax = plt.subplots(1,1)
for n in range(1,150,10):
    df = data[data["Cycle number"]==N1[n]]
    df = df.dropna()
    df["Time [s]"] = df["Time [s]"] - df["Time [s]"].iloc[0]
    df["Q [Ah]"] = df["Q [Ah]"] - df["Q [Ah]"].iloc[0]
    t = df["Time [s]"].to_numpy()
    I = df["Current [mA]"].to_numpy()/1000
    V = df["Voltage [V]"].to_numpy()
    Q = df["Q [Ah]"].to_numpy()
    Cap = df["Capacity [Ah]"].to_numpy()
    E = df["Expansion [mu m]"].to_numpy()
    E = E-E[0]
    idxi1 = np.where((np.diff(I)<-1) & (I[:-1]>0) & (I[:-1]<2))[0]
    idx = idxi1[0]
    t_ch = t[:idx-1]
    I_ch = I[:idx-1]
    V_ch = V[:idx-1]
    Q_ch = Cap[:idx-1]
    E_ch = E[:idx-1]
    t_dh = t[idx+1:]-t[idx+1]
    I_dh = I[idx+1:]
    V_dh = V[idx+1:]
    Q_dh = Cap[idx+1:]
    E_dh = E[idx+1:]

    window_length=31
    polyorder=3
    S_ch = Q_ch/max(Q_ch)
    dQ = savgol_filter(Q_ch,window_length,polyorder,1)
    dE2 = savgol_filter(E_ch,window_length,polyorder,2)
    dE2 = dE2*(S_ch<=0.95)
    dEdQ2 = dE2/dQ/dQ
    pks_,_ = find_peaks(-dEdQ2,height=0.01,width=0.1)
    pks1 = [pk for pk in pks_ if S_ch[pk]>=0.1 and S_ch[pk]<=0.9]
    pks_,_ = find_peaks(dEdQ2,height=0.01,width=0.1)
    pks2 = [pk for pk in pks_ if S_ch[pk]>=0.1 and S_ch[pk]<=0.9]
    pks = np.append(pks1,pks2)

    p1s = np.round(S_ch[pks1],4)
    p2s = np.round(S_ch[pks2],4)
    p1q = np.round(Q_ch[pks1],3)
    p2q = np.round(Q_ch[pks2],3)
    Qmax = np.round(Q_ch[-1],3)
    
# ax.plot(t_ch,E_ch)
    ax.plot(S_ch,dEdQ2)
    ax.set_ylim([-40,40])
# df1 = pd.DataFrame({"N":n+1,"p1s":p1s,"p2s":p2s,"p1q":p1q,"p2q":p2q,"Qch":Qmax},index=[0])

In [ ]:
cell = 28
n = 16
cell_no = f'{cell:02d}'
data = pd.read_csv(cyc_DIR+"cycling_data_cell_"+cell_no+".csv")
N1 = np.unique(data["Cycle number"])
# print(N1[-1])
df = data[data["Cycle number"]==N1[n]]
df = df.dropna()
df["Time [s]"] = df["Time [s]"] - df["Time [s]"].iloc[0]
df["Q [Ah]"] = df["Q [Ah]"] - df["Q [Ah]"].iloc[0]
t = df["Time [s]"].to_numpy()
I = df["Current [mA]"].to_numpy()/1000
V = df["Voltage [V]"].to_numpy()
Q = df["Q [Ah]"].to_numpy()
Cap = df["Capacity [Ah]"].to_numpy()
E = df["Expansion [mu m]"].to_numpy()
E = E-E[0]
idxi1 = np.where((np.diff(I)<-1) & (I[:-1]>0) & (I[:-1]<2))[0]
idx = idxi1[0]
t_ch = t[:idx-1]
I_ch = I[:idx-1]
V_ch = V[:idx-1]
Q_ch = Cap[:idx-1]
E_ch = E[:idx-1]
t_dh = t[idx+1:]-t[idx+1]
I_dh = I[idx+1:]
V_dh = V[idx+1:]
Q_dh = Cap[idx+1:]
E_dh = E[idx+1:]

window_length=51
polyorder=3
S_ch = Q_ch/max(Q_ch)
dQ = savgol_filter(Q_ch,window_length,polyorder,1)
dE2 = savgol_filter(E_ch,window_length,polyorder,2)
dE2 = dE2*(S_ch<=0.95)
dEdQ2 = dE2/dQ/dQ
pks_,_ = find_peaks(-dEdQ2,prominence=0.1)
pks1 = [pk for pk in pks_ if S_ch[pk]>=0.1 and S_ch[pk]<=0.9]
pks_,_ = find_peaks(dEdQ2,prominence=0.1)
pks2 = [pk for pk in pks_ if S_ch[pk]>=0.1 and S_ch[pk]<=0.9]
pks = np.append(pks1,pks2)

p1s = np.round(S_ch[pks1],4)
p2s = np.round(S_ch[pks2],4)
p1q = np.round(Q_ch[pks1],3)
p2q = np.round(Q_ch[pks2],3)
Qmax = np.round(Q_ch[-1],3)

fig,ax = plt.subplots(1,1,figsize=(5,4))
# ax1 = ax.flat[0]
# ax1.plot(t,E)
# ax2 = ax.flat[1]
# ax2.plot(t,I)
ax.plot(Q_ch,dEdQ2)
ax.plot(Q_ch[pks1],dEdQ2[pks1],'ro',markersize=10)
ax.plot(Q_ch[pks2],dEdQ2[pks2],'ro',markersize=10)
ax.text(1.8,-20,r"$x_{p1}=0.24$",fontsize=15)
ax.text(3,15,r"$x_{p2}=0.5$",fontsize=15)
# ax.set_xlim([0.1,0.9])
ax.set_ylim([-25,+25])
ax.set_title(r"2nd Derivative of Expansion wrt Q: $\frac{d^2t}{dQ^2}$")
ax.set_xlabel("Q [Ah]")
ax.set_ylabel(r"$\frac{d^2t}{dQ^2}$")
fig.savefig(fig_DIR + f"Model_Expansion_data.png")
# df1 = pd.DataFrame({"N":n+1,"p1s":p1s,"p2s":p2s,"p1q":p1q,"p2q":p2q,"Qch":Qmax},index=[0])

In [ ]:
cvbvc

In [ ]:
window_length=51
polyorder=3
dQ = savgol_filter(Q_ch,window_length,polyorder,1)
dV = savgol_filter(V_ch,window_length,polyorder,1)
dVdQ = dV/dQ
fig,ax = plt.subplots(1,1)
ax.plot(Q_ch/max(Q_ch),dVdQ)
ax.set_xlim([0.1,0.9])
ax.set_ylim([0,0.6])

In [ ]:
def graphite_volume_change(sto):
    stoichpoints = np.array([0,0.12,0.18,0.24,0.50,1])
    thicknesspoints = np.array([0,2.406/100,3.3568/100,4.3668/100,5.583/100,13.0635/100])
    x = [sto]
    t_change = pybamm.Interpolant(stoichpoints, thicknesspoints, x, name=None, interpolator='linear', extrapolate=True, entries_string=None)
    t_change = np.interp(x,stoichpoints,thicknesspoints)
    return t_change

In [ ]:
xa = np.arange(0,1,0.001)
En = []
for x in xa:
    En.append(graphite_volume_change(x))
En = np.concatenate(En)

In [ ]:
window_length=271
polyorder=3
dQ = savgol_filter(xa,window_length,polyorder,1)
dE2 = savgol_filter(En,window_length,polyorder,2)
dEdQ2 = dE2/dQ/dQ

In [ ]:
fig, ax = plt.subplots(2,1,figsize=(7,8))
ax1 = ax.flat[0]
ax1.plot(xa,En)
ax1.set_title("Volumetric Strain")
ax2 = ax.flat[1]
ax2.plot(xa,dEdQ2)
ax2.set_title("2nd Diff Volumetric Strain")
ax2.axvline(x=0.24,color='r', linestyle='--',linewidth=1,clip_on=False,ymin=0,ymax=2)
ax2.axvline(x=0.50,color='r', linestyle='--',linewidth=1,clip_on=False,ymin=0,ymax=2)
ax1.text(0.25,0.08,r"$x_{p1}=0.24$")
ax1.text(0.51,0.08,r"$x_{p2}=0.5$")
ax2.text(0.25,0,r"$x_{p1}=0.24$")
ax2.text(0.51,0,r"$x_{p2}=0.5$")
ax2.set_xlabel("Stoichiometry")
fig.savefig(fig_DIR + f"Model_Expansion.png")